# Exploring proteomics data for SPT analysis
Data release 9 (August 2026) has proteomics data for about 10k subjects.

Quality report: https://support.researchallofus.org/hc/en-us/articles/50655639562900-All-of-Us-Genomics-Multi-omics-Quality-Report

Data dictionaries: https://support.researchallofus.org/hc/en-us/articles/360033200232-Data-Dictionaries

Multiomic organization: https://support.researchallofus.org/hc/en-us/articles/49999549117588-How-the-All-of-Us-Genomic-and-mulit-omics-data-are-organized

The goal is to:
1. Check which SPT proteins are present in the proteomics dataset
2. Define PD and Control subjects
3. Extract SPTSSB carriers from chr3
4. Compare the levels of SPT proteins between SPTSSB PD risk variant carriers vs. non-carriers (main analysis: Control subjects)

## 1. Checking SPT proteins available

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls gs://vwb-aou-datasets-controlled/v9/multiomics/proteomics

In [ ]:
# gcloud synthax is a bit different
!gcloud storage cat \
  gs://vwb-aou-datasets-controlled/v9/multiomics/proteomics/manifest.tsv \
  --billing-project=$GOOGLE_PROJECT | head

In [ ]:
!gcloud storage --billing-project=$GOOGLE_PROJECT ls gs://vwb-aou-datasets-controlled/v9/multiomics/proteomics/npx/

# We provide NPX data in Parquet and tsv format. NPX values are derived from raw data counts
# after a subtraction and normalization pipeline, to minimize technical intra- and inter-plate
# variation. The log₂ scale values demonstrate the biological protein expression for differential
# expression analysis.

# The NPX tsv and Parquets have the same information, but are in different formats depending on
# your downstream usage. Parquet format is a columnar binary storage format for data retrieval
# and file storage efficiency. Parquet format can be used downstream with cloud pipelines and
# use tools within Python, R, or Spark.

In [ ]:
# Use .tsv for a random subject and identify all proteins detected.

!gcloud storage --billing-project=$GOOGLE_PROJECT head gs://vwb-aou-datasets-controlled/v9/multiomics/proteomics/npx/tsv/AoU_Proteomics_LCSET-29041__Extended_NPX_2024-01-25.tsv
# Assay The gene symbol or common name of the specific protein target being measured
# (e.g., IL6, TNF)

In [ ]:
# !gcloud storage cp \
#   gs://vwb-aou-datasets-controlled/v9/multiomics/proteomics/npx/tsv/AoU_Proteomics_LCSET-29041__Extended_NPX_2024-01-25.tsv \
#   ./workspace/Pipelines_and_files/proteomics/proteomics_test_LCSET-29041.tsv \
#   --billing-project=$GOOGLE_PROJECT

import os
import pandas as pd

os.chdir("./workspace/Pipelines_and_files/proteomics")

prot = pd.read_csv("proteomics_test_LCSET-29041.tsv", sep="\t")

prot.columns

In [ ]:
# Get clean protein list
proteins = (
    prot[["Assay", "UniProt", "OlinkID"]]
    .drop_duplicates()
    .sort_values("Assay")
)

proteins

In [ ]:
# SPT partial match
proteins[proteins["Assay"].str.contains("SPT", case=False, na=False)]

In [ ]:
# Also look at ARS proteins, since I'm working with ARSA as well
proteins[proteins["Assay"].str.contains("ARS", case=False, na=False)]

## 2. Define PD and Control subjects

Code borrowed/modified from Ruth_PD_GWAS_data_prep.ipynb

In [ ]:
os.environ["WORKSPACE_CDR"] = "wb-silky-artichoke-2408.C2025Q4R6" # v9; v8 would be wb-silky-artichoke-2408.C2024Q3R8
print(os.environ["WORKSPACE_CDR"])

In [ ]:
# This query represents dataset "PD_clinical-sex_age" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_PD_person_sql = """
SELECT
    person.person_id,
    person.birth_datetime AS date_of_birth,
    p_sex_at_birth_concept.concept_name AS sex_at_birth
FROM
    `""" + os.environ["WORKSPACE_CDR"] + """.person` person
LEFT JOIN
    `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept
    ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
WHERE
    person.person_id IN (
        SELECT DISTINCT person_id
        FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
        WHERE
            cb_search_person.person_id IN (
                SELECT person_id
                FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person`
                WHERE has_whole_genome_variant = 1
            )
            AND cb_search_person.person_id IN (
                SELECT criteria.person_id
                FROM (
                    SELECT DISTINCT
                        person_id,
                        entry_date,
                        concept_id
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events`
                    WHERE
                        concept_id IN (
                            SELECT DISTINCT c.concept_id
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c
                            JOIN (
                                SELECT CAST(cr.id AS STRING) AS id
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr
                                WHERE
                                    concept_id IN (381270)
                                    AND full_text LIKE '%_rank1]%'
                            ) a
                                ON (
                                    c.path LIKE CONCAT('%.', a.id, '.%')
                                    OR c.path LIKE CONCAT('%.', a.id)
                                    OR c.path LIKE CONCAT(a.id, '.%')
                                    OR c.path = a.id
                                )
                            WHERE
                                is_standard = 1
                                AND is_selectable = 1
                        )
                        AND is_standard = 1
                ) criteria
            )
    )
"""

PD_clinical = pd.read_gbq(
    dataset_PD_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

PD_clinical.head()

In [ ]:
# Seems like the old SQL pipeline to query v8 data does not work for v9...
# https://support.researchallofus.org/hc/en-us/articles/41981213271188-Data-Explorer-in-Researcher-Workbench

# Let's try to debug

test_sql = f"""
SELECT COUNT(*) AS n
FROM `{os.environ["WORKSPACE_CDR"]}.person`
"""

pd.read_gbq(test_sql)

In [ ]:
# Check whether old PD concept is present in v9

import os
import pandas as pd

test_pd_sql = f"""
SELECT
    concept_id,
    concept_name,
    domain_id,
    vocabulary_id,
    concept_code
FROM `{os.environ["WORKSPACE_CDR"]}.concept`
WHERE concept_id = 381270
"""

pd.read_gbq(test_pd_sql)

In [ ]:
pd_sql = f"""
SELECT DISTINCT
    p.person_id,
    p.birth_datetime AS date_of_birth,
    sex.concept_name AS sex_at_birth

FROM `{os.environ["WORKSPACE_CDR"]}.person` p

JOIN `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` co
    ON p.person_id = co.person_id

JOIN `{os.environ["WORKSPACE_CDR"]}.concept_ancestor` ca
    ON co.condition_concept_id = ca.descendant_concept_id

LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` sex
    ON p.sex_at_birth_concept_id = sex.concept_id

WHERE ca.ancestor_concept_id = 381270
"""

PD_clinical = pd.read_gbq(
    pd_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

print("PD subjects:", len(PD_clinical))
PD_clinical.head()

In [ ]:
# Double-check what diagnoses were captured by the query above

check_sql = f"""
SELECT
    c.concept_id,
    c.concept_name,
    COUNT(DISTINCT co.person_id) AS n_people

FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` co

JOIN `{os.environ["WORKSPACE_CDR"]}.concept_ancestor` ca
    ON co.condition_concept_id = ca.descendant_concept_id

JOIN `{os.environ["WORKSPACE_CDR"]}.concept` c
    ON co.condition_concept_id = c.concept_id

WHERE ca.ancestor_concept_id = 381270

GROUP BY c.concept_id, c.concept_name
ORDER BY n_people DESC
"""

pd.read_gbq(check_sql)

In [ ]:
PD_clinical['sex_at_birth'].value_counts()

In [ ]:
# Now, querying old individuals
# age between 70 and 124, using the same birthday-aware age calculation;
# exclude anyone with an entry in the death table.

age70_sql = f"""
SELECT
    p.person_id,
    gender.concept_name AS gender,
    p.birth_datetime AS date_of_birth,
    sex.concept_name AS sex_at_birth

FROM `{os.environ["WORKSPACE_CDR"]}.person` p

LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` gender
    ON p.gender_concept_id = gender.concept_id

LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` sex
    ON p.sex_at_birth_concept_id = sex.concept_id

WHERE
    DATE_DIFF(CURRENT_DATE(), DATE(p.birth_datetime), YEAR)
    - IF(
        EXTRACT(MONTH FROM DATE(p.birth_datetime)) * 100
        + EXTRACT(DAY FROM DATE(p.birth_datetime))
        >
        EXTRACT(MONTH FROM CURRENT_DATE()) * 100
        + EXTRACT(DAY FROM CURRENT_DATE()),
        1,
        0
    )
    BETWEEN 70 AND 124

    AND NOT EXISTS (
        SELECT 1
        FROM `{os.environ["WORKSPACE_CDR"]}.death` d
        WHERE d.person_id = p.person_id
    )
"""

age70_df = pd.read_gbq(
    age70_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

print("Participants age 70+:", len(age70_df))
age70_df.head()

In [ ]:
control_exclusion_sql = f"""
SELECT DISTINCT
    p.person_id

FROM `{os.environ["WORKSPACE_CDR"]}.person` p

JOIN `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` co
    ON p.person_id = co.person_id

WHERE

    -- Standard concepts + descendants
    co.condition_concept_id IN (
        SELECT descendant_concept_id
        FROM `{os.environ["WORKSPACE_CDR"]}.concept_ancestor`
        WHERE ancestor_concept_id IN (
            43531003,
            4101305,
            378144,
            373747
        )
    )

    OR

    -- Non-standard/source concepts + descendants
    co.condition_source_concept_id IN (
        SELECT descendant_concept_id
        FROM `{os.environ["WORKSPACE_CDR"]}.concept_ancestor`
        WHERE ancestor_concept_id IN (
            1568287,
            1568284,
            1568286,
            35207355,
            1568289,
            35207329
        )
    )
"""

control_exclusion = pd.read_gbq(
    control_exclusion_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

print("Participants for control exclusion:", len(control_exclusion))
control_exclusion.head()

In [ ]:
# Sanity check

concept_ids = [
    43531003, 4101305, 378144, 373747,
    1568287, 1568284, 1568286,
    35207355, 1568289, 35207329
]

concept_check_sql = f"""
SELECT
    concept_id,
    concept_name,
    domain_id,
    vocabulary_id,
    concept_code,
    standard_concept
FROM `{os.environ["WORKSPACE_CDR"]}.concept`
WHERE concept_id IN ({','.join(map(str, concept_ids))})
ORDER BY standard_concept DESC, concept_name
"""

concept_check = pd.read_gbq(concept_check_sql)
concept_check

In [ ]:
descendant_check_sql = f"""
SELECT
    ancestor_concept_id,
    COUNT(DISTINCT descendant_concept_id) AS n_descendants
FROM `{os.environ["WORKSPACE_CDR"]}.concept_ancestor`
WHERE ancestor_concept_id IN ({','.join(map(str, concept_ids))})
GROUP BY ancestor_concept_id
ORDER BY ancestor_concept_id
"""

pd.read_gbq(descendant_check_sql)

In [ ]:
# This query represents dataset "All_control-sex_age" for domain "person" and was generated for All of Us Controlled Tier Dataset v8 (should be the same for v9)
controls_sql = f"""
SELECT
    p.person_id,
    p.birth_datetime AS date_of_birth,
    sex.concept_name AS sex_at_birth

FROM `{os.environ["WORKSPACE_CDR"]}.person` p

LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` sex
    ON p.sex_at_birth_concept_id = sex.concept_id
"""

controls_notfiltered = pd.read_gbq(
    controls_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

print("Total participants:", len(controls_notfiltered))
controls_notfiltered.head()

In [ ]:
controls_filtered = controls_notfiltered[
    ~controls_notfiltered['person_id'].isin(control_exclusion['person_id'])
]
controls_filtered.shape[0]

In [ ]:
controls_filtered['sex_at_birth'].value_counts()

In [ ]:
controls_filtered_sex = controls_filtered[
    controls_filtered['sex_at_birth'].isin(['Female','Male'])
]
controls_filtered_sex.shape[0]

In [ ]:
PD_clinical_sex = PD_clinical[
    PD_clinical['sex_at_birth'].isin(['Female','Male'])
]
PD_clinical_sex.shape[0]

In [ ]:
PD_clinical_sex.dtypes

In [ ]:
PD_clinical_sex['PD'] = '1'
controls_filtered_sex['PD'] = '0'
PHENO_DF = pd.concat([PD_clinical_sex, controls_filtered_sex])

In [ ]:
PHENO_DF['BIRTH_YEAR'] = PHENO_DF['date_of_birth'].dt.year
PHENO_DF['AGE_COV'] = 2024 - PHENO_DF['BIRTH_YEAR']
PHENO_DF['AGE_COV_SQUARED'] = PHENO_DF['AGE_COV']**2
PHENO_DF.head()

In [ ]:
PHENO_DF[
    PHENO_DF['PD'] == '1'
]['AGE_COV'].median(),PHENO_DF[
    PHENO_DF['PD'] == '0'
]['AGE_COV'].median()

In [ ]:
PHENO_DF.head()

## NOTE THAT race and ethnicity here are self-reported. We do not really use those as biological proxies; 
# instead, we use genetic ancestry.

In [ ]:
PHENO_DF.shape[0]

In [ ]:
PHENO_DF.to_csv('PHENO_DF_v9.txt', sep = '\t', index = None)
# this file was moved to one folder above workspace directory, to avoid loss when instance is terminated

In [ ]:
#os.getenv("WORKSPACE_BUCKET")
#!echo $WORKSPACE_BUCKET
!gcloud storage buckets list

In [ ]:
# Save important files to GCS bucket
!gsutil -u $GOOGLE_PROJECT cp PHENO_DF_v9.txt gs://pipelines-and-files-wb-prompt-kiwi-3077/

## 3. Extract SPTSSB carriers from chr3

Use rs1450522 as the PD GWAS proxy.

In [ ]:
# First, find v9 PLINK path
!gcloud storage ls \
    gs://vwb-aou-datasets-controlled/v9/ \
    --billing-project=$GOOGLE_PROJECT

In [ ]:
!gcloud storage ls \
    gs://vwb-aou-datasets-controlled/v9/** \
    --billing-project=$GOOGLE_PROJECT | grep -i plink

In [ ]:
# Check .bim to know whether rs1450522 is genotyped
!gcloud storage cat \
    gs://vwb-aou-datasets-controlled/v9/microarray/plink/arrays.bim \
    --billing-project=$GOOGLE_PROJECT \
    | head

In [ ]:
!gcloud storage cat \
    gs://vwb-aou-datasets-controlled/v9/microarray/plink/arrays.bim \
    --billing-project=$GOOGLE_PROJECT \
    | grep 'chr3:161359842:'

In [ ]:
# Since variant is not genotyped, we need to use short read WGS

# Download plink2
!wget https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
!unzip plink2_linux_x86_64_latest.zip
!chmod +x plink2

In [ ]:
!echo -e 'chr3:161359842:A' > SPTSSB_rs1450522.snplist

In [ ]:
!gcloud storage ls \
    gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen \
    --billing-project=$GOOGLE_PROJECT

In [ ]:
!gcloud storage cat \
    gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr3.pvar \
    --billing-project=$GOOGLE_PROJECT \
    | grep 'chr3:161359842:'

In [ ]:
!./plink2 \
    --pfile ~/workspace/vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr3 \
    --extract SPTSSB_rs1450522.snplist \
    --export A \
    --out SPTSSB_rs1450522

In [ ]:
!head SPTSSB_rs1450522.raw

In [ ]:
geno = pd.read_csv(
    "SPTSSB_rs1450522.raw",
    sep=r"\s+"
)

geno["rs1450522_status"] = geno["chr3:161359842:A_A"].map({
    0: "NC",
    1: "Het",
    2: "Hom"
})

geno['rs1450522_status'].value_counts()

In [ ]:
# Counting the correct effect allele (which is G)
geno["G_count"] = 2 - geno["chr3:161359842:A_A"]

geno["rs1450522_status"] = geno["G_count"].map({
    0: "NC",
    1: "Het",
    2: "Hom"
})

geno["rs1450522_status"].value_counts()

In [ ]:
# Saving file
geno.to_csv('SPTSSB_rs1450522_shortWGS_carriers_v9.txt', sep = '\t', index = None)

!gsutil -u $GOOGLE_PROJECT cp SPTSSB_rs1450522_shortWGS_carriers_v9.txt gs://pipelines-and-files-wb-prompt-kiwi-3077/

## 4. Levels of SPT proteins in SPTSSB carriers

Only SPTLC1 is detected by the olink platform, so I will focus on that.

In [ ]:
# Need to merge PD status (PHENO_DF)/rs1450522 status (geno)/SPTLC1 proteomics data (prot)

PHENO_DF.head()

In [ ]:
geno.head()

In [ ]:
import os
import pandas as pd

os.chdir("./workspace/Pipelines_and_files/proteomics")

prot = pd.read_csv("proteomics_test_LCSET-29041.tsv", sep="\t")
prot.head()

In [ ]:
# Prepare phenotype dataframe
pheno_sub = PHENO_DF[
    ["person_id", "PD", "AGE_COV", "sex_at_birth"]
].copy()

pheno_sub = pheno_sub.rename(columns={
    "person_id": "IID",
    "sex_at_birth": "SEX"
})

pheno_sub["IID"] = pheno_sub["IID"].astype(str)

In [ ]:
# Prepare genotype dataframe
geno_sub = geno[
    ["IID", "rs1450522_status","G_count"]
].copy()

geno_sub["IID"] = geno_sub["IID"].astype(str)

In [ ]:
# Extract person_id from beginning of SampleID
prot["IID"] = prot["SampleID"].astype(str).str.split("_").str[0]

# Keep only SPTLC1
sptlc1 = prot[
    prot["Assay"].str.upper().eq("SPTLC1")
].copy()

# Keep only necessary columns
sptlc1 = sptlc1[
    ["IID", "NPX"]
].copy()

sptlc1 = sptlc1.rename(columns={
    "NPX": "SPTLC1_NPX"
})

In [ ]:
# Merge everything
df_final = (
    pheno_sub
    .merge(geno_sub, on="IID", how="inner")
    .merge(sptlc1, on="IID", how="inner")
)

df_final.head()

In [ ]:
# Sanity checks
print(df_final.shape)

print("\nPD status:")
print(df_final["PD"].value_counts(dropna=False))

print("\nGenotype status:")
print(df_final["rs1450522_status"].value_counts(dropna=False))

print("\nPD x genotype:")
print(pd.crosstab(
    df_final["PD"],
    df_final["rs1450522_status"],
    margins=True
))

print("\nSex:")
print(df_final["SEX"].value_counts(dropna=False))

In [ ]:
df_final["SPTLC1_NPX"].describe()

In [ ]:
# Plotting basic graphs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import seaborn as sns

In [ ]:
# First, define the genotype order and dosage
df = df_final.copy()

geno_order = ["NC", "Het", "Hom"]

df["rs1450522_status"] = pd.Categorical(
    df["rs1450522_status"],
    categories=geno_order,
    ordered=True
)

df["G_count"] = df["rs1450522_status"].map({
    "NC": 0,
    "Het": 1,
    "Hom": 2
}).astype(float)

In [ ]:
df.head()

## 4.1 Uncorrected boxplot

In [ ]:
# For controls only
d = df[df["PD"] == "0"].copy()

fig, ax = plt.subplots(figsize=(6, 5))

data = [
    d.loc[d["rs1450522_status"] == g, "SPTLC1_NPX"].dropna()
    for g in geno_order
]

ax.boxplot(data, tick_labels=geno_order)

# Individual subjects
for i, g in enumerate(geno_order, start=1):
    y = d.loc[d["rs1450522_status"] == g, "SPTLC1_NPX"].dropna()
    x = np.random.normal(i, 0.04, size=len(y))
    ax.scatter(x, y, alpha=0.5, s=18)

ax.set_xlabel("rs1450522 genotype")
ax.set_ylabel("SPTLC1 NPX (log2)")
ax.set_title("SPTLC1 abundance — controls")

plt.tight_layout()
plt.show()

In [ ]:
# Additive linear regression
model_raw = smf.ols(
    "SPTLC1_NPX ~ G_count",
    data=d
).fit()

print(model_raw.summary())

## 4.2 Corrected boxplot (by age and sex)

In [ ]:
model_adj = smf.ols(
    "SPTLC1_NPX ~ G_count + AGE_COV + C(SEX)",
    data=d
).fit()

print("Adjusted beta per G allele:", model_adj.params["G_count"])
print("Adjusted P-value:", model_adj.pvalues["G_count"])
print("95% CI:",
      model_adj.conf_int().loc["G_count"].tolist())

In [ ]:
beta_adj = model_adj.params["G_count"]

print("Adjusted fold-change per G allele:", 2**beta_adj)
print("Adjusted percent change:", (2**beta_adj - 1) * 100)

In [ ]:
cov_model = smf.ols(
    "SPTLC1_NPX ~ AGE_COV + C(SEX)",
    data=d
).fit()

d["SPTLC1_NPX_adjusted"] = (
    cov_model.resid + d["SPTLC1_NPX"].mean()
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

data = [
    d.loc[d["rs1450522_status"] == g, "SPTLC1_NPX_adjusted"].dropna()
    for g in geno_order
]

ax.boxplot(data, tick_labels=geno_order)

for i, g in enumerate(geno_order, start=1):
    y = d.loc[
        d["rs1450522_status"] == g,
        "SPTLC1_NPX_adjusted"
    ].dropna()

    x = np.random.normal(i, 0.04, size=len(y))
    ax.scatter(x, y, alpha=0.5, s=18)

ax.set_xlabel("rs1450522 genotype")
ax.set_ylabel("Age/sex-adjusted SPTLC1 NPX")
ax.set_title("SPTLC1 abundance — controls")

plt.tight_layout()
plt.show()